In [2]:
# CRAG , LangGraph

import os
from dotenv import load_dotenv
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_db = Chroma(persist_directory="./chroma_db_session", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# 1. State 정의: 노드 간에 전달될 데이터 구조
class GraphState(TypedDict):
    question: str
    documents: List[str]
    generation: str
    web_search_needed: bool

# 2. Node 정의
def retrieve(state: GraphState):
    print("  -> [Node] Retrieve: 문서 검색 중...")
    question = state["question"]
    docs = retriever.invoke(question)
    return {"documents": [d.page_content for d in docs], "question": question}

def grade_documents(state: GraphState):
    print("  -> [Node] Grade: 검색된 문서의 관련성 자체 평가 중...")
    question = state["question"]
    documents = state["documents"]
    
    prompt = ChatPromptTemplate.from_template(
        "다음 문서가 사용자의 질문과 관련이 있는지 평가하세요.\n"
        "관련이 있으면 'yes', 없으면 'no'라고만 답하세요.\n\n"
        "문서: {context}\n질문: {question}"
    )
    chain = prompt | llm | StrOutputParser()
    
    filtered_docs = []
    web_search_needed = False
    
    for idx, doc in enumerate(documents):
        score = chain.invoke({"context": doc, "question": question}).strip().lower()
        if "yes" in score:
            print(f"     - 문서 {idx+1}: [PASS] 관련성 높음")
            filtered_docs.append(doc)
        else:
            print(f"     - 문서 {idx+1}: [FAIL] 관련성 낮음 (환각 위험)")
            web_search_needed = True
            
    if not filtered_docs:
        web_search_needed = True
        
    return {"documents": filtered_docs, "web_search_needed": web_search_needed}